# Reasoning — GSM8K and Big-Bench Hard (local)

Two classic multi-step reasoning benchmarks, both answered with the model's own step-by-step working:

- **GSM8K** — 1,319 grade-school maths word problems (8-shot chain of thought).
- **BBH (Big-Bench Hard)** — 27 tasks, ~6,500 questions, that stumped earlier models: logical deduction, tracking shuffled objects, date arithmetic, sports and causal judgement, parenthesis matching, and more (zero-shot chain of thought).

They complement `run-math` (competition mathematics) with everyday numerical reasoning and a broad spread of logic puzzles. The score is the average of the two.

This notebook runs the benchmarks **locally**, straight against the model's chat endpoint — no EvalHub and no OpenShift. All settings are in the configuration cell below.

**How it works:** [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) (`lm_eval`) is run through `uvx`, once per benchmark. Questions go to the model's `/chat/completions` endpoint with the chat template applied.

**Requires:** [`uv`](https://docs.astral.sh/uv/) (for `uvx`) and network access to the model endpoint and Hugging Face (for the datasets). The full run is long (~7,800 questions with step-by-step answers); start with `LIMIT = 5`.

## 1. Configuration

Everything you might want to change lives in this one cell.

In [ ]:
import json, os, shutil, subprocess
from getpass import getpass
from pathlib import Path

# --- Model endpoint (from .env, or prompted for if there is no .env) ---
def load_env():
    """Read KEY=VALUE lines from a .env file into os.environ. Returns False if there is no file.

    Looks in the kernel's working directory, then ./notebooks (some Jupyter servers start the
    kernel in the repo's parent folder rather than next to the notebook)."""
    p = next((c for c in (Path(".env"), Path("notebooks/.env")) if c.is_file()), None)
    if p is None:
        return False
    for line in p.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip("\"'")
    return True

if load_env():
    print("Loaded settings from .env")
    missing = [k for k in ("OPENAI_BASE_URL", "OPENAI_MODEL") if not os.environ.get(k)]
    if missing:
        raise SystemExit(f"Please set {', '.join(missing)} in .env (see dot_env.template)")
else:
    print("No .env file found (copy dot_env.template to .env to skip these prompts)")
    os.environ["OPENAI_BASE_URL"] = input("Model base URL (e.g. https://host/v1): ").strip()
    os.environ["OPENAI_MODEL"] = input("Model name: ").strip()
    os.environ["OPENAI_API_KEY"] = getpass("API key (leave blank if none): ")

MODEL_URL = os.environ["OPENAI_BASE_URL"].rstrip("/")  # OpenAI-compatible base URL
MODEL_NAME = os.environ["OPENAI_MODEL"]
MODEL_API_KEY = os.environ.get("OPENAI_API_KEY", "")


# --- Proxy (optional; set PROXY_URL in .env, leave empty for a direct connection) ---
def apply_proxy():
    """Export the proxy settings in the form each tool underneath reads them.

    uv, Python (requests / huggingface_hub / aiohttp) and npm read HTTP(S)_PROXY and NO_PROXY
    (upper or lower case); Node's built-in fetch needs NODE_USE_ENV_PROXY; npm also has its own
    npm_config_* names. CA_BUNDLE is for proxies that re-sign HTTPS traffic with a company CA."""
    url, no_proxy = os.environ.get("PROXY_URL", "").strip(), os.environ.get("NO_PROXY", "").strip()
    if url:
        # The model endpoint is sent through the proxy too, unless its host is listed in NO_PROXY.
        for k in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy", "npm_config_proxy", "npm_config_https_proxy"):
            os.environ[k] = url
        os.environ["NODE_USE_ENV_PROXY"] = "1"
        if no_proxy:
            for k in ("NO_PROXY", "no_proxy", "npm_config_noproxy"):
                os.environ[k] = no_proxy
        print("Using proxy", url.split("@")[-1], "(NO_PROXY: " + (no_proxy or "none") + ")")
        # lm-eval 0.4.12 sends concurrent requests with aiohttp and never sets trust_env, so aiohttp
        # would ignore the variables above. This sitecustomize.py (loaded at start-up of the lm_eval
        # process via PYTHONPATH) makes every aiohttp.ClientSession trust them.
        patch_dir = Path(".eval-notebook-proxy-patch").resolve()
        patch_dir.mkdir(exist_ok=True)
        (patch_dir / "sitecustomize.py").write_text(
            "try:\n"
            "    import aiohttp\n"
            "    _init = aiohttp.ClientSession.__init__\n"
            "    def __init__(self, *a, **kw):\n"
            "        kw.setdefault('trust_env', True)\n"
            "        _init(self, *a, **kw)\n"
            "    aiohttp.ClientSession.__init__ = __init__\n"
            "except ImportError:\n"
            "    pass\n")
        os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(patch_dir), os.environ.get("PYTHONPATH")]))
    ca = os.environ.get("CA_BUNDLE", "").strip()
    if ca:
        ca = str(Path(ca).expanduser().resolve())
        for k in ("SSL_CERT_FILE", "REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE", "NODE_EXTRA_CA_CERTS"):
            os.environ[k] = ca
        print("Using CA bundle", ca)

apply_proxy()


# --- Benchmark ---
GSM8K_TASK = "gsm8k_cot"            # lm-eval task names
BBH_TASK = "bbh_cot_zeroshot"
LM_EVAL_VERSION = "0.4.12"  # pinned so results are repeatable
LM_EVAL_EXTRAS = "api"  # pip extras lm-eval needs for these tasks
PYPI_INDEX = "https://pypi.org/simple"  # package index used to install lm-eval, instead of any index set in your uv config
NUM_CONCURRENT = 8          # parallel requests to the model
THRESHOLD = 0.6  # minimum average score to count as PASS
MAX_GEN_TOKS = 2048         # max tokens per answer; the harness defaults (256 / 1024) cut off long working before the answer
LIMIT = None                # per task (BBH has 27), e.g. 5 for a quick smoke test; None runs everything

## Helper

`run()` launches a command and streams its output into the notebook as it arrives, so you
can watch progress. It raises an error if the command exits non-zero.

In [ ]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with {proc.returncode}")

## 2. Run the benchmarks

Runs GSM8K, then BBH. Completed requests are cached on disk, so if a run dies partway, re-running this cell resumes where it left off.

In [ ]:
def lm_eval(task, *extra):
    """Run one lm-evaluation-harness task and return the folder holding its results and samples.

    Completed requests are cached, so if the run dies partway, re-running resumes where it stopped;
    the cache is deleted after a successful run so the next run gets fresh answers from the model."""
    output = Path(f".eval-notebook-{task}")
    cache = Path(f".eval-notebook-{task}-cache") / "requests"
    cache.parent.mkdir(exist_ok=True)
    shutil.rmtree(output, ignore_errors=True)
    cmd = ["uvx", "--default-index", PYPI_INDEX, "--python", "3.12", "--from", f"lm-eval[{LM_EVAL_EXTRAS}]=={LM_EVAL_VERSION}",
           "lm_eval", "--model", "local-chat-completions",
           "--model_args", f"model={MODEL_NAME},base_url={MODEL_URL.rstrip('/')}/chat/completions,"
                           f"num_concurrent={NUM_CONCURRENT},max_retries=5,tokenized_requests=False",
           "--tasks", task, "--apply_chat_template", "--log_samples",
           "--output_path", str(output), "--use_cache", str(cache), *extra]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    # Clear uv index overrides inherited from the environment so lm-eval installs from PYPI_INDEX.
    uv_env = {k: "" for k in ("UV_INDEX", "UV_INDEX_URL", "UV_EXTRA_INDEX_URL", "UV_DEFAULT_INDEX")}
    run(cmd, env={**uv_env, "OPENAI_API_KEY": MODEL_API_KEY or "not-required"})
    shutil.rmtree(cache.parent, ignore_errors=True)
    return output

GSM_OUTPUT = lm_eval(GSM8K_TASK, "--gen_kwargs", f"max_gen_toks={MAX_GEN_TOKS}")
BBH_OUTPUT = lm_eval(BBH_TASK, "--gen_kwargs", f"max_gen_toks={MAX_GEN_TOKS}")

## 3. Score and pass/fail

Each benchmark is scored on the model's final answer (`flexible-extract`, which reads the answer wherever it appears). The headline is the average of the two. Per-question answers are in the `.eval-notebook-<task>` folders (`samples_*.jsonl`).

In [ ]:
def headline(output, task):
    results = json.loads(next(output.rglob("results_*.json")).read_text())["results"]
    return results[task]["exact_match,flexible-extract"]

gsm, bbh = headline(GSM_OUTPUT, GSM8K_TASK), headline(BBH_OUTPUT, BBH_TASK)
score = (gsm + bbh) / 2
print(f"GSM8K {gsm:.3f}   BBH {bbh:.3f}")
print("PASS" if score >= THRESHOLD else "FAIL", f"- average {score:.3f} vs threshold {THRESHOLD}")

## 4. Results dashboard

Lenient vs. strict scoring, accuracy against the length of the model's working, all 27 BBH tasks ranked with 95% confidence intervals, and examples of problems it missed.

In [ ]:
import sys
from pathlib import Path
# eval_report.py sits next to the notebooks (the kernel may have started in the parent folder)
sys.path.insert(0, str(next(d for d in (Path("."), Path("notebooks")) if (d / "eval_report.py").is_file())))
from eval_report import report_reasoning

report_reasoning(GSM_OUTPUT, BBH_OUTPUT, THRESHOLD)